In [ ]:
from time import sleep as delay
from project.el5600 import project
if "ic" in globals():
    ic.close()
ic = project(device="el5600", revision="a1", emulator="cp2112", logging=False, is_gui=False)

from phy.multimeter import siglent_sdm3055_auto
from phy.power_supply import keithley_2460, rigol_dp821a, keysight_e36232a
from phy.scope import tektronix_dpo4104b_usb
from phy.eloader import sdl1020x
from phy.relay_16ch import relay_box
# from phy.chamber_th3 import chamber

%matplotlib tk
from interface.docs.output_chart import plot
from interface.cui_colors import color
from interface.i2c_bridge.tca9548a import tca9548
from interface.docs.output_excel import excel_frame, style
from interface.cui_logger import logger as log


import numpy as np

chart = plot()

dm = siglent_sdm3055_auto()
ps = rigol_dp821a()
ks = keysight_e36232a()
kt = keithley_2460()
ld = sdl1020x()
sc = tektronix_dpo4104b_usb()

# dma = agilent_34401a("COM5")
relay = relay_box(i2c_h=ic)
# tc = chamber(port=3)
# mux = tca9548(ic.i2c_h, 0x70)


# ==================================
def disable_all_ps(kt=kt, ps=ps):
    kt.disable
    ld.disable
    ks.disable
    ps.ch1.disable
    ps.ch2.disable
# ==================================

# Test A14

ks : ---> dm.ch1 (I_VIN) ---> relay.ch1 (VIN)

kt : VDD3V3

dm.ch1 : ---> relay.ch1 (I) ---> relay.ch1 (VIN)

dm.ch2 : ---> relay.ch1 (VIN) and GND

dm.ch3 : ---> relay.ch2 (VOUT) and GND

dm.ch4 : ---> temperature

ps.ch1 : relay

ps.ch2 : ---> relay.ch3 (EN)

relay.ch1 : VIN

relay.ch2 : VOUT

relay.ch3 : EN

In [ ]:
log.output_set_filename(f"TSD")
log.output_csv(["TEMP (C)", "VIN (V)", "VOUT (V)", "EN (V)", "I_VIN (uA)", "SW_STS", "TSD_STS", "OFF_BY_TSD", "TSD_AUTO_ON_EN"])

In [ ]:
disable_all_ps()
relay.init_channels


v_vdd = 3.3
v_en = 3.3

# --------------------------------------------
ps.ch1.cfg_all = 5, 1
ps.ch1.enable

# --------------------------------------------
kt.cfg_all = v_vdd, 0.01 # vdd
kt.enable
delay(0.5)

# --------------------------------------------
ps.ch2.cfg_all = v_en, 0.1
ps.ch2.enable
delay(0.5)

relay.ch3.enable # en
delay(0.5)

# --------------------------------------------
ks.cfg_all = 20, 0.1
ks.power_recycle
relay.enable = 1, 2

# --------------------------------------------
ic.TSD_AUTO_ON_EN = 1

In [ ]:
ic.status

In [ ]:
count = 0

while True:
    
    try:
        sw_sts = ic.SW_STS
        tsd_sts = ic.TSD_STS
        off_by_tsd = ic.OFF_BY_TSD
        tsd_auto_on_en = ic.TSD_AUTO_ON_EN
        
    except:
        sw_sts = "NACK"
        tsd_sts = "NACK"
        off_by_tsd = "NACK"
        tsd_auto_on_en = "NACK"
    
    m_iin  = f"{dm.ch1.current_2mA * 1e+6:.06f}"
    m_vin  = f"{dm.ch2.voltage_200V:.06f}"
    m_vout = f"{dm.ch3.voltage_200V:.06f}"
    m_temp = f"{dm.ch4.temperature:.02f}"
    
    ret = [m_temp, m_vin, m_vout, v_en, m_iin, sw_sts, tsd_sts, off_by_tsd, tsd_auto_on_en]
    log.output_csv(ret)
    
    print(f"{m_temp}, {m_vin}, {m_vout}, {v_en}, {m_iin}, {sw_sts}, {tsd_sts}, {off_by_tsd}, {tsd_auto_on_en}")
    
    if sw_sts != 2:
        count += 1
    
    if count == 10:
        break

In [ ]:
count = 0

while True:
    
    try:
        sw_sts = ic.SW_STS
        tsd_sts = ic.TSD_STS
        off_by_tsd = ic.OFF_BY_TSD
        tsd_auto_on_en = ic.TSD_AUTO_ON_EN
        
    except:
        sw_sts = "NACK"
        tsd_sts = "NACK"
        off_by_tsd = "NACK"
        tsd_auto_on_en = "NACK"
    
    m_iin  = f"{dm.ch1.current_2mA * 1e+6:.06f}"
    m_vin  = f"{dm.ch2.voltage_200V:.06f}"
    m_vout = f"{dm.ch3.voltage_200V:.06f}"
    m_temp = f"{dm.ch4.temperature:.02f}"
    
    ret = [m_temp, m_vin, m_vout, v_en, m_iin, sw_sts, tsd_sts, off_by_tsd, tsd_auto_on_en]
    log.output_csv(ret)
    
    print(f"{m_temp}, {m_vin}, {m_vout}, {v_en}, {m_iin}, {sw_sts}, {tsd_sts}, {off_by_tsd}, {tsd_auto_on_en}")
    
    if sw_sts == 2:
        count += 1
    
    if count == 20:
        break

# ----------------------------------------------------------------------------

relay.init_channels
disable_all_ps()

# debugging

In [ ]:
log.output_set_filename(f"TSD - timing")
log.output_csv(["Time (s)", "TEMP (C)", "VIN (V)", "VOUT (V)", "EN (V)", "I_VIN (uA)", "SW_STS", "TSD_STS", "OFF_BY_TSD", "TSD_AUTO_ON_EN"])

count = 0

from time import time

lap_start = time()

while True:
    
    try:
        sw_sts = ic.SW_STS
        tsd_sts = ic.TSD_STS
        off_by_tsd = ic.OFF_BY_TSD
        tsd_auto_on_en = ic.TSD_AUTO_ON_EN
        
    except:
        sw_sts = "NACK"
        tsd_sts = "NACK"
        off_by_tsd = "NACK"
        tsd_auto_on_en = "NACK"
    
    m_iin  = round(dm.ch1.current_2mA, 6)
    m_vin  = round(dm.ch2.voltage_200V, 6)
    m_vout = round(dm.ch3.voltage_200V, 6)
    m_temp = round(dm.ch4.temperature, 2)
    
    lap_time = round(time() - lap_start, 1)
    
    ret = [lap_time, m_temp, m_vin, m_vout, v_en, m_iin, sw_sts, tsd_sts, off_by_tsd, tsd_auto_on_en]
    log.output_csv(ret)
    
    print(f"{lap_time}, {m_temp}, {m_vin}, {m_vout}, {v_en}, {m_iin}, {sw_sts}, {tsd_sts}, {off_by_tsd}, {tsd_auto_on_en}")
    
    if sw_sts != 2:
        count += 1
    
    if count == 20:
        break


In [ ]:
log.output_set_filename(f"TSD - timing, falling")
log.output_csv(["Time (s)", "TEMP (C)", "VIN (V)", "VOUT (V)", "EN (V)", "I_VIN (uA)", "SW_STS", "TSD_STS", "OFF_BY_TSD", "TSD_AUTO_ON_EN"])

count = 0
from time import time
lap_start = time()
while True:
    
    try:
        sw_sts = ic.SW_STS
        tsd_sts = ic.TSD_STS
        off_by_tsd = ic.OFF_BY_TSD
        tsd_auto_on_en = ic.TSD_AUTO_ON_EN
        
    except:
        sw_sts = "NACK"
        tsd_sts = "NACK"
        off_by_tsd = "NACK"
        tsd_auto_on_en = "NACK"
    
    m_iin  = round(dm.ch1.current_2mA, 6)
    m_vin  = round(dm.ch2.voltage_200V, 6)
    m_vout = round(dm.ch3.voltage_200V, 6)
    m_temp = round(dm.ch4.temperature, 2)
    
    lap_time = round(time() - lap_start, 1)
    
    ret = [lap_time, m_temp, m_vin, m_vout, v_en, m_iin, sw_sts, tsd_sts, off_by_tsd, tsd_auto_on_en]
    log.output_csv(ret)
    
    print(f"{lap_time}, {m_temp}, {m_vin}, {m_vout}, {v_en}, {m_iin}, {sw_sts}, {tsd_sts}, {off_by_tsd}, {tsd_auto_on_en}")
    
    if sw_sts == 2:
        count += 1
    
    if count == 20:
        break

# ----------------------------------------------------------------------------

relay.init_channels
disable_all_ps()

In [ ]:
ic.status